# Reuso do pipeline em outro cenário — Segmentação de doadores (RFM)

Este notebook não toca em `rfm_pipeline.py`. RFM não é uma técnica exclusiva
de e-commerce — times de fundraising de ONGs usam exatamente a mesma lógica
(Recência = dias desde a última doação, Frequência = nº de doações
distintas, Monetário = valor total doado) para decidir quem convidar para o
programa de grandes doadores versus quem precisa de uma campanha de
reengajamento.

Dados sintéticos gerados abaixo só para demonstração.

In [1]:
import numpy as np
import pandas as pd

from rfm_pipeline import clean_transactions, compute_rfm, flag_outliers_iqr, select_k, fit_segments, profile_segments

rng = np.random.default_rng(7)
n_donors = 800
n_donations = 6000

donor_ids = rng.integers(9000, 9000 + n_donors, size=n_donors)
profile = rng.choice(["major_donor", "regular", "one_time"], size=n_donors, p=[0.08, 0.42, 0.50])
donor_profile = dict(zip(donor_ids, profile))

start = pd.Timestamp("2023-01-01")
end = pd.Timestamp("2024-12-31")
days_range = (end - start).days

rows = []
receipt_counter = 100000
for _ in range(n_donations):
    donor = rng.choice(donor_ids)
    prof = donor_profile[donor]
    receipt_counter += 1

    base_amount = {"major_donor": 5000, "regular": 150, "one_time": 60}[prof]
    amount = round(max(5, rng.normal(base_amount, base_amount * 0.5)), 2)
    date = start + pd.Timedelta(days=int(rng.integers(0, days_range)))

    rows.append({
        "ReceiptID": str(receipt_counter),
        "FundCode": rng.choice(["10001", "10002A", "10003"]),
        "DonorID": donor,
        "Amount": amount,
        "DonationDate": date,
    })

donations_df = pd.DataFrame(rows)
donations_df.head()

,ReceiptID,FundCode,DonorID,Amount,DonationDate
0,100001,10001,9515,5.00,2023-03-03
1,100002,10003,9515,166.44,2023-12-28
2,100003,10002A,9134,58.17,2024-11-14
3,100004,10003,9755,57.54,2024-10-11
4,100005,10002A,9784,34.35,2024-07-11


## 1. Mesma limpeza, mesma função, domínio diferente

Aqui não há faturas canceladas nem stock codes — mas a função ainda serve:
basta apontar `invoice_col` para o identificador de transação (`ReceiptID`)
e `stockcode_col` para qualquer campo categórico que precise de validação de
formato (aqui, `FundCode`). Nenhum registro deveria ser descartado neste
dataset sintético — e o relatório abaixo confirma isso quantitativamente,
em vez de simplesmente assumir.

In [2]:
cleaned, report = clean_transactions(
    donations_df,
    invoice_col="ReceiptID",
    stockcode_col="FundCode",
    customer_col="DonorID",
    quantity_col="Amount",  # nao ha quantidade negativa em doacoes; serve so como checagem de sanidade
    price_col="Amount",
    invoice_pattern=r"^\d+$",
    stockcode_patterns=(r"^\d{5}$", r"^\d{5}[A-Z]+$"),
)
report

,reason,rows_dropped,pct_of_original
0,invalid_invoice_format,0,0.0
1,invalid_stockcode,0,0.0
2,missing_customer_id,0,0.0
3,non_positive_price,0,0.0
4,TOTAL_KEPT,6000,100.0
5,TOTAL_DROPPED,0,0.0


## 2. Mesmo `compute_rfm`, trocando só os nomes de coluna

In [3]:
rfm = compute_rfm(
    cleaned,
    customer_col="DonorID",
    invoice_col="ReceiptID",
    date_col="DonationDate",
    amount_col="Amount",
)
rfm.describe()

,DonorID,Monetary,Frequency,LastPurchaseDate,Recency
count,502.000000,502.000000,502.000000,502,502.000000
mean,9404.928287,5360.910020,11.952191,2024-10-18 22:16:43.984063,72.071713
min,9002.000000,86.680000,2.000000,2023-09-26 00:00:00,0.000000
25%,9197.250000,567.365000,7.000000,2024-09-27 00:00:00,19.000000
50%,9411.500000,979.185000,10.000000,2024-11-10 12:00:00,49.500000
75%,9607.750000,1936.060000,16.000000,2024-12-11 00:00:00,94.000000
max,9799.000000,171757.970000,41.000000,2024-12-30 00:00:00,461.000000
std,233.874710,18501.611217,6.891351,NaN,74.731287


## 3. Mesmo pipeline de outliers, k, fit e nomeação

In [4]:
non_outliers, outliers = flag_outliers_iqr(rfm, monetary_col="Monetary", frequency_col="Frequency")
print(f"Não-outliers: {len(non_outliers)} | Outliers: {len(outliers)}")

best_k, search = select_k(non_outliers, feature_cols=["Monetary", "Frequency", "Recency"], log_cols=["Monetary", "Frequency"])
print(f"k recomendado: {best_k} (silhouette={search['silhouette'].max():.3f})")

segmented = fit_segments(non_outliers, outliers, feature_cols=["Monetary", "Frequency", "Recency"],
                          log_cols=["Monetary", "Frequency"], n_clusters=best_k)
segmented["SegmentName"].value_counts()

Não-outliers: 453 | Outliers: 49


k recomendado: 3 (silhouette=0.396)


SegmentName
Champions                             208
Loyal                                 189
Potential Loyalist                     56
Pamper (big spender, infrequent)       37
Upsell (frequent, low ticket)           7
Delight (top-tier: big & frequent)      5
Name: count, dtype: int64

In [5]:
profile = profile_segments(segmented)
profile

,n_customers,avg_monetary,avg_frequency,avg_recency,total_revenue,pct_of_customers,pct_of_revenue
SegmentName,,,,,,,
"Pamper (big spender, infrequent)",37,44866.17,14.32,60.11,1660048.33,7.4,61.7
Delight (top-tier: big & frequent),5,98327.72,33.00,12.20,491638.58,1.0,18.3
Champions,208,1787.30,16.12,42.86,371757.82,41.4,13.8
Loyal,189,644.93,7.34,60.46,121892.31,37.6,4.5
Potential Loyalist,56,518.38,5.71,240.21,29029.44,11.2,1.1
"Upsell (frequent, low ticket)",7,2401.48,34.71,14.43,16810.35,1.4,0.6


## 4. Leitura de negócio

Os nomes técnicos (`Champions`, `Pamper`, etc.) continuam genéricos o
suficiente para fazer sentido fora do varejo, mas o time de fundraising
normalmente traduziria: `Pamper` (grande doador esporádico) vira candidato
a "major gift officer" dedicado; `Champions`/`Loyal` viram a lista de
doadores recorrentes para o programa de reconhecimento anual.

O que prova reuso aqui não é o nome do segmento — é que **zero linhas de
`rfm_pipeline.py` mudaram** entre o notebook de varejo e este. Só os
parâmetros de nome de coluna.